# Huplet deformation energy

This notebook illustrates the companion note [`húplet-deformation-energy.typ`](../../further_steps/húplet-deformation-energy.typ). For the two-frequency profile

$$F_k(x)=(1-k)\cos(2\pi n x)+k\cos(2\pi d x),\qquad 0\le k\le1,$$

it follows the $n$ labeled crest branches $X_i(k)$ that begin at $i/n$. The main plot is the labeled squared-displacement statistic

$$E(k)=\sum_{i=0}^{n-1}\delta_i(k)^2,\qquad \delta_i(k)=X_i(k)-\frac{i}{n}.$$

This is a deformation measure, not physical signal energy. Labels and the continuous lifts are retained; no rotation or relabeling is minimized away.

In [5]:
from fractions import Fraction
from math import gcd

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from scipy.optimize import brentq

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Numerical and plotting dependencies loaded.")

Numerical and plotting dependencies loaded.


## Following the labeled crests

The notebook assumes the companion paper's standing hypotheses: $1<n<d$, $\gcd(n,d)=1$, and $n$ odd. Define the nearest endpoint-grid index

$$m_i=\left\lfloor\frac{di}{n}+\frac12\right\rfloor.$$

Odd $n$ rules out rounding ties. The theorem places $X_i(k)$ in the short corridor between $i/n$ and $m_i/d$, where the stationarity equation

$$G_k(x)=(1-k)n\sin(2\pi n x)+kd\sin(2\pi d x)=0$$

has exactly one root. Solving inside these disjoint corridors is more reliable than selecting the highest peaks of a sampled curve: it preserves each branch label even when residual crests are born elsewhere.

In [6]:
def validate_pair(n, d):
    """Validate the hypotheses used by the odd-n continuation theorem."""
    if not isinstance(n, (int, np.integer)) or not isinstance(d, (int, np.integer)):
        raise ValueError("n and d must be integers.")
    if not 1 < n < d:
        raise ValueError("Expected 1 < n < d.")
    if n % 2 == 0:
        raise ValueError("This notebook treats the odd-n case only.")
    if gcd(int(n), int(d)) != 1:
        raise ValueError("n and d must be coprime.")


def nearest_endpoint_index(i, n, d):
    """Exact integer form of floor(di/n + 1/2)."""
    return (2 * d * i + n) // (2 * n)


def stationarity(x, k, n, d):
    """F'_k with the irrelevant common factor -2π removed."""
    return (1.0 - k) * n * np.sin(2 * np.pi * n * x) + k * d * np.sin(2 * np.pi * d * x)


def crest_curvature_factor(x, k, n, d):
    """Positive when a stationary point is a nondegenerate crest."""
    return (1.0 - k) * n**2 * np.cos(2 * np.pi * n * x) + k * d**2 * np.cos(2 * np.pi * d * x)


def anchored_position(i, k, n, d):
    """Return the lifted branch X_i(k) from its certified corridor."""
    anchor = i / n
    endpoint = nearest_endpoint_index(i, n, d) / d

    if i == 0 or k <= 0.0:
        return anchor
    if k >= 1.0:
        return endpoint

    left, right = sorted((anchor, endpoint))
    return brentq(
        stationarity, left, right, args=(k, n, d),
        xtol=5e-15, rtol=1e-13, maxiter=100,
    )


def exact_energy(n, d, *, balanced=False):
    """Exact E at k=1, or at k_c=n/(n+d) when balanced=True."""
    total = Fraction(0, 1)
    for i in range(n):
        m_i = nearest_endpoint_index(i, n, d)
        if balanced:
            position = Fraction(i + m_i, n + d)
        else:
            position = Fraction(m_i, d)
        total += (position - Fraction(i, n)) ** 2
    return total


def compute_deformation(n, d, *, points=401):
    """Compute all lifted positions, displacements, and E(k)."""
    validate_pair(n, d)
    k_c = n / (n + d)
    k_values = np.unique(np.r_[np.linspace(0.0, 1.0, points), k_c])
    positions = np.empty((len(k_values), n), dtype=float)

    for column, i in enumerate(range(n)):
        positions[:, column] = [anchored_position(i, k, n, d) for k in k_values]

    anchors = np.arange(n, dtype=float) / n
    displacements = positions - anchors
    energy = np.sum(displacements**2, axis=1)
    curvature = crest_curvature_factor(positions, k_values[:, None], n, d)
    if np.min(curvature) <= -1e-9:
        raise RuntimeError("A computed branch failed the crest check.")

    return {
        "n": n, "d": d, "k_c": k_c, "k": k_values,
        "positions": positions, "displacements": displacements,
        "energy": energy, "curvature": curvature,
        "E_balanced_exact": exact_energy(n, d, balanced=True),
        "E_endpoint_exact": exact_energy(n, d),
    }


print("Certified-corridor continuation helpers loaded.")

Certified-corridor continuation helpers loaded.


## Interactive energy and branch displacements

Choose odd $n$ from the dropdown, then move the $d$ slider to animate the plot. Whenever $n$ changes, the $d$ slider is rebuilt from the coprime integers larger than that $n$. The upper panel shows $E(k)$ (or $E(k)/n$); the lower panel shows the individual signed displacements.

The vertical scales are fixed across every selectable $(n,d)$ pair, making changes in magnitude directly visible. The dashed line is the slope-balanced parameter $k_c=n/(n+d)$. The two marked energy values are arithmetically exact: at $k_c$, $X_i=(i+m_i)/(n+d)$, and at $k=1$, $X_i=m_i/d$.

In [7]:
def fraction_text(value):
    return str(value.numerator) if value.denominator == 1 else f"{value.numerator}/{value.denominator}"


CONTROL_N_MIN, CONTROL_N_MAX = 3, 31
CONTROL_D_MAX = 80
ODD_N_VALUES = tuple(range(CONTROL_N_MIN, CONTROL_N_MAX + 1, 2))
SELECTABLE_PAIRS = tuple(
    (n, d)
    for n in ODD_N_VALUES
    for d in range(n + 1, CONTROL_D_MAX + 1)
    if gcd(n, d) == 1
)

# Every branch stays inside its anchor-to-endpoint corridor. Therefore the
# largest exact endpoint values over the control range give fixed global scales.
FIXED_RAW_ENERGY_MAX = 1.08 * float(max(
    exact_energy(n, d) for n, d in SELECTABLE_PAIRS
))
FIXED_NORMALIZED_ENERGY_MAX = 1.08 * float(max(
    exact_energy(n, d) / n for n, d in SELECTABLE_PAIRS
))
FIXED_DISPLACEMENT_MAX = 1.08 * float(max(
    abs(Fraction(nearest_endpoint_index(i, n, d), d) - Fraction(i, n))
    for n, d in SELECTABLE_PAIRS
    for i in range(n)
))


def make_energy_figure(n, d, *, normalized=False, points=401):
    data = compute_deformation(n, d, points=points)
    k_values = data["k"]
    k_c = data["k_c"]
    scale = n if normalized else 1
    plotted_energy = data["energy"] / scale
    exact_balanced = data["E_balanced_exact"] / scale
    exact_endpoint = data["E_endpoint_exact"] / scale
    balanced_index = int(np.argmin(np.abs(k_values - k_c)))

    fig, (energy_ax, displacement_ax) = plt.subplots(
        2, 1, figsize=(10, 7.2), sharex=True,
        gridspec_kw={"height_ratios": (1.25, 1.0), "hspace": 0.08},
    )

    energy_ax.plot(k_values, plotted_energy, color="#6a3d9a", linewidth=2.4)
    energy_ax.fill_between(k_values, 0.0, plotted_energy, color="#cab2d6", alpha=0.28)
    energy_ax.scatter(
        [k_c, 1.0], [float(exact_balanced), float(exact_endpoint)],
        color=["#ff7f00", "#6a3d9a"], s=42, zorder=4,
    )
    energy_ax.axvline(k_c, color="0.35", linestyle="--", linewidth=1.0)
    energy_ax.set_ylabel(r"$E(k)/n$" if normalized else r"$E(k)$")
    energy_ax.set_ylim(
        0.0, FIXED_NORMALIZED_ENERGY_MAX if normalized else FIXED_RAW_ENERGY_MAX
    )
    energy_ax.grid(axis="y", color="0.9", linewidth=0.8)
    energy_ax.text(
        0.02, 0.96,
        rf"$E(k_c)={fraction_text(exact_balanced)}\approx{float(exact_balanced):.6g}$" + "\n"
        + rf"$E(1)={fraction_text(exact_endpoint)}\approx{float(exact_endpoint):.6g}$",
        transform=energy_ax.transAxes, ha="left", va="top", fontsize=9,
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "0.85", "alpha": 0.92},
    )

    colors = plt.cm.viridis(np.linspace(0.08, 0.92, n))
    for i, color in enumerate(colors):
        displacement_ax.plot(
            k_values, data["displacements"][:, i],
            color=color, linewidth=1.15, label=rf"$\delta_{{{i}}}$",
        )
        displacement_ax.scatter(
            [1.0], [data["displacements"][-1, i]],
            color=[color], s=13, zorder=3,
        )

    displacement_ax.axhline(0.0, color="0.3", linewidth=0.8)
    displacement_ax.axvline(k_c, color="0.35", linestyle="--", linewidth=1.0)
    displacement_ax.set(
        xlim=(0.0, 1.0),
        ylim=(-FIXED_DISPLACEMENT_MAX, FIXED_DISPLACEMENT_MAX),
        xlabel=r"$k$", ylabel=r"$\delta_i(k)$",
    )
    displacement_ax.set_xticks([0.0, k_c, 1.0], [r"$0$", rf"$k_c={k_c:.4f}$", r"$1$"])
    displacement_ax.grid(axis="y", color="0.92", linewidth=0.8)
    if n <= 11:
        displacement_ax.legend(
            frameon=False, ncol=min(n, 6), loc="upper center",
            bbox_to_anchor=(0.5, -0.24), fontsize=8,
        )

    fig.suptitle(rf"Huplet deformation: $(n,d)=({n},{d})$", fontsize=14)
    fig.subplots_adjust(top=0.92, bottom=0.16 if n <= 11 else 0.09)

    data["balanced_index"] = balanced_index
    return fig, data


DEFAULT_N, DEFAULT_D = 5, 7

if WIDGETS_AVAILABLE:
    def valid_d_values(n):
        return tuple(d for d in range(n + 1, CONTROL_D_MAX + 1) if gcd(n, d) == 1)

    n_control = widgets.Dropdown(
        options=ODD_N_VALUES, value=DEFAULT_N, description="odd n",
        layout={"width": "180px"},
    )
    d_control = widgets.SelectionSlider(
        options=valid_d_values(DEFAULT_N), value=DEFAULT_D, description="d",
        continuous_update=True,
        layout={"width": "700px"},
    )
    normalized_control = widgets.Checkbox(
        value=False, description="show E(k)/n", indent=False,
        layout={"width": "160px"},
    )
    plot_output = widgets.Output()
    control_update_in_progress = False

    def redraw(change=None):
        if control_update_in_progress:
            return
        with plot_output:
            clear_output(wait=True)
            try:
                fig, current = make_energy_figure(
                    n_control.value, d_control.value,
                    normalized=normalized_control.value,
                )
            except ValueError as error:
                print(error)
                return
            display(fig)
            plt.close(fig)
            minimum_increment = np.min(np.diff(current["energy"]))
            print(
                f"Exact endpoint E(1) = {fraction_text(current['E_endpoint_exact'])}; "
                f"smallest sampled energy increment = {minimum_increment:.3e}."
            )

    def rebuild_d_slider(change):
        global control_update_in_progress
        control_update_in_progress = True
        try:
            previous_d = d_control.value
            choices = valid_d_values(change["new"])
            replacement = min(
                choices, key=lambda candidate: (abs(candidate - previous_d), candidate)
            )
            d_control.options = choices
            d_control.value = replacement
        finally:
            control_update_in_progress = False
        redraw()

    n_control.observe(rebuild_d_slider, names="value")
    for control in (d_control, normalized_control):
        control.observe(redraw, names="value")

    controls = widgets.VBox([n_control, d_control, normalized_control])
    display(widgets.VBox([controls, plot_output]))
    redraw()
else:
    print("ipywidgets is unavailable; displaying the default pair (5, 7).")
    figure, _ = make_energy_figure(DEFAULT_N, DEFAULT_D)
    plt.show()

## What to look for

- $E(0)=0$ because every labeled branch begins at its anchor.
- Reflected labels have opposite displacements, so their squared contributions coincide. The fixed branch $X_0(k)=0$ contributes zero.
- The balanced marker often exposes visible structure, but the plot is evidence rather than a proof of any qualitative claim.
- At $k=1$, the displayed value is checked against exact nearest-grid rounding, not merely read from the numerical curve.

The lower panel is essential context: a collective statistic can hide how individual labeled crests move.

## Reproducibility checks

These checks exercise the default pair, including the exact balanced and endpoint configurations, stationarity, crest status, and reflection symmetry.

In [8]:
check = compute_deformation(5, 7, points=151)
n, d = check["n"], check["d"]
balanced_index = int(np.argmin(np.abs(check["k"] - check["k_c"])))

expected_endpoint = np.array([nearest_endpoint_index(i, n, d) / d for i in range(n)])
expected_balanced = np.array([
    (i + nearest_endpoint_index(i, n, d)) / (n + d) for i in range(n)
])

assert np.allclose(check["positions"][0], np.arange(n) / n)
assert np.allclose(check["positions"][-1], expected_endpoint)
assert np.allclose(check["positions"][balanced_index], expected_balanced, atol=2e-12)
assert np.isclose(check["energy"][-1], float(check["E_endpoint_exact"]), atol=2e-14)
assert np.isclose(check["energy"][balanced_index], float(check["E_balanced_exact"]), atol=2e-14)
assert np.min(check["curvature"]) > 0.0

residual = stationarity(
    check["positions"], check["k"][:, None], n, d,
)
reflection_error = max(
    np.max(np.abs(check["displacements"][:, i] + check["displacements"][:, n - i]))
    for i in range(1, n)
)

assert np.max(np.abs(residual)) < 2e-10
assert reflection_error < 2e-12

print(
    "All checks passed for (n,d)=(5,7).\n"
    f"max |G_k(X_i)| = {np.max(np.abs(residual)):.3e}\n"
    f"minimum crest factor = {np.min(check['curvature']):.6g}\n"
    f"max reflection error = {reflection_error:.3e}"
)

All checks passed for (n,d)=(5,7).
max |G_k(X_i)| = 5.398e-12
minimum crest factor = 13.1356
max reflection error = 2.381e-14
